# Ablation Row 4 — + Hard negative mining

| | Value |
|---|---|
| Features | full (same as rows 2-3) |
| PCA | 4677 components |
| Scales | (0.75, 1.0, 1.5) + cross-scale NMS |
| HNM | **4 rounds** (loaded from cached run; converged 11508 → 581 → 753 → 809) |
| Threshold | `cfg.detection_threshold = 1.5` (default — row 5 will tune this) |

This notebook **loads** the post-HNM model from `microglia-artifacts-row4/`
(copied from the 2026-05-22 refactored pipeline run, which executed the exact
same HNM logic: full features → fit-once scaler+PCA → 4 rounds with fixed C,
colour-fix, fit-once design).

**Test evaluation only — no retraining.** Saves ~70 min vs running HNM from
scratch. If you ever need to rerun HNM, restore the original cell set from git
history.


## 1  Imports + Config

In [ ]:
import os
import joblib
import numpy as np
import matplotlib.image as mpimg

from pipeline import (
    Config,
    extract_raw_features, fit_pipeline,
    tune_svm, train_svm, evaluate_classifier,
    process_image, multi_scale_detect, non_max_suppression,
    load_gt_boxes, evaluate_detections, plot_gt_vs_pred,
    save_hard_negatives, tune_detection_threshold,
    ensure_test_patches, patch_level_test_eval, detection_level_test_eval,
    extract_features, list_images,
)

%matplotlib inline

In [ ]:
cfg = Config(
    artifact_dir='./microglia-artifacts-row4',
    feature_mode='full',
    scale_factors=(0.75, 1.0, 1.5),
    pca_n_components=4677,
    detection_threshold=1.5,
)
os.makedirs(cfg.artifact_dir, exist_ok=True)
os.makedirs(cfg.hnm_folder, exist_ok=True)
print(cfg)

## 2  Load HNM-trained model

In [ ]:
# Load the post-HNM svm_clf + scaler + pca that were saved by the prior
# fit-once HNM run. cfg.artifact_dir points at the row-4 copy; row 5 will use
# this same dir for threshold tuning.
svm_clf = joblib.load(cfg.svm_clf_path)
scaler  = joblib.load(cfg.scaler_path)
pca     = joblib.load(cfg.pca_path)

print(f"Loaded HNM-trained model from {cfg.artifact_dir}/")
print(f"  svm_clf : C={svm_clf.C}, coef shape={svm_clf.coef_.shape}, n_iter={getattr(svm_clf, 'n_iter_', '?')}")
print(f"  scaler  : n_features_in_={scaler.n_features_in_}")
print(f"  pca     : n_components_={pca.n_components_}, "
      f"variance_retained={pca.explained_variance_ratio_.sum():.1%}")

## 3  Held-out test evaluation (default threshold = 1.5)

In [ ]:
ensure_test_patches(cfg)

print("── Patch-level test ──")
patch_level_test_eval(svm_clf, scaler, pca, cfg)

print("\n── Detection-level test (multi-scale, untuned t=%.2f) ──" % cfg.detection_threshold)
row4_metrics = detection_level_test_eval(svm_clf, scaler, pca, cfg, show_plots=True)